In [1]:
# morpho

In [5]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Running this notebook on: ", device)

import spateo as st
print("Last run with spateo version:", st.__version__)

# Other imports
import warnings, string
import anndata as ad
import dynamo as dyn
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Uncomment the following if running on the server
import pyvista as pv
pv.start_xvfb()
from matplotlib.colors import ListedColormap, rgb2hex
sns.set_theme(context="paper", style="ticks", font_scale=1)
warnings.filterwarnings('ignore')
# %load_ext autoreload
# %autoreload 2

Running this notebook on:  cpu
Last run with spateo version: 0.0.0


In [6]:
adata = sc.read_h5ad('/data/work/05.cluster/FuseMap/0106/Hippocampus_latent_embeddings_all_single_pretrain/dmt_leiden_20250108_1.h5ad')
adata.obs_names_make_unique()
adata

AnnData object with n_obs × n_vars = 1112773 × 33326
    obs: 'dnbCount', 'area', 'orig.ident', 'x', 'y', 'region', 'n_counts', 'region_h2', 'Tangram_1119_celltype', 'Tangram_1119_celltype_main_frac', 'region_hip', 'slice_code', 'sub_region', 'dmt_leiden', 'dmt_leiden_merge', 'dmt_leiden_anno'
    uns: 'dmt_leiden_colors', 'dmt_nn', 'leiden', 'slice_code_colors'
    obsm: 'X_dmt', 'X_dmt_highdim', 'align_spatial_2d', 'align_spatial_3d', 'cell_border', 'latent_embeddings_all_single_pretrain', 'latent_embeddings_all_spatial_pretrain', 'spatial', 'spatial_division'
    obsp: 'dmt_nn_connectivities', 'dmt_nn_distances'

In [7]:

dic = {'0': 'T7_SST_LHX_SIX3',
 '1': 'T9_CRH',
 '10': 'T6_CLSTN2',
 '11': 'T4_COL4A1_vessel',
 '12': 'T3_BMP4_IGFBP5',
 '13': 'T7_SST_LHX_SIX3',
 '14': 'T0_PCP4_VGF',
 '15': 'T9_CRH',
 '16': 'T3_BMP4_IGFBP5',
 '17': 'T3_BMP4_IGFBP5',
 '18': 'T2_PRSS12',
 '19': 'T9_CRH',
 '2': 'T9_CRH',
 '20': 'T1_NTS_CALB1',
 '21': 'T1_NTS_CALB1',
 '22': 'T1_NTS_CALB1',
 '23': 'T10_HBZ',
 '24': 'T1_NTS_CALB1',
 '25': 'T1_NTS_CALB1',
 '26': 'T2_PRSS12',
 '27': 'T9_CRH',
 '28': 'T2_PRSS12',
 '29': 'T5_AQP4_gial_region',
 '3': 'T10_HBZ',
 '30': 'T8_CBLN4_CBLN1',
 '31': 'T7_SST_LHX_SIX3',
 '32': 'T9_CRH',
 '33': 'T1_NTS_CALB1',
 '34': 'T7_SST_LHX_SIX3',
 '35': 'T10_HBZ',
 '36': 'T8_CBLN4_CBLN1',
 '37': 'T9_CRH',
 '38': 'T3_BMP4_IGFBP5',
 '39': 'T1_NTS_CALB1',
 '4': 'T2_PRSS12',
 '40': 'T9_CRH',
 '41': 'T6_CLSTN2',
 '42': 'T10_HBZ',
 '5': 'T7_SST_LHX_SIX3',
 '6': 'T7_SST_LHX_SIX3',
 '7': 'T9_CRH',
 '8': 'T2_PRSS12',
 '9': 'T1_NTS_CALB1'}

adata.obs['dmt_leiden_annotation_0115'] = [dic[i] for i in adata.obs['dmt_leiden']]
colormap = {'T2_PRSS12': '#31d6d3',
 'T6_CLSTN2': '#6b0c4d',
 'T3_BMP4_IGFBP5': '#e94a1d',
 'T4_COL4A1_vessel': '#cf58e5',
 'T10_HBZ': '#39d789',
 'T7_SST_LHX_SIX3': '#4e7c26',
 'T8_CBLN4_CBLN1': '#eaccd8',
 'T1_NTS_CALB1': '#9114fb',
 'T9_CRH': '#79f4ec',
 'T0_PCP4_VGF': '#b3fcdd',
 'T5_AQP4_gial_region': '#455edf'}

In [9]:
adata = adata[:, ~adata.var.index.str.startswith('HB')]
adata = adata[:, ~adata.var.index.str.startswith('RP')]
adata = adata[:, ~adata.var.index.str.startswith('MT-')]
adata

View of AnnData object with n_obs × n_vars = 1112773 × 33153
    obs: 'dnbCount', 'area', 'orig.ident', 'x', 'y', 'region', 'n_counts', 'region_h2', 'Tangram_1119_celltype', 'Tangram_1119_celltype_main_frac', 'region_hip', 'slice_code', 'sub_region', 'dmt_leiden', 'dmt_leiden_merge', 'dmt_leiden_anno', 'dmt_leiden_annotation_0115'
    uns: 'dmt_leiden_colors', 'dmt_nn', 'leiden', 'slice_code_colors'
    obsm: 'X_dmt', 'X_dmt_highdim', 'align_spatial_2d', 'align_spatial_3d', 'cell_border', 'latent_embeddings_all_single_pretrain', 'latent_embeddings_all_spatial_pretrain', 'spatial', 'spatial_division'
    obsp: 'dmt_nn_connectivities', 'dmt_nn_distances'

In [17]:
GW10_adata = adata[adata.obs['slice_code'] == 'A03587A5C6_WT2024071215080.h5ad'].copy()
GW10_adata.obs['stage'] = 'GW10'
GW10_adata.obs.index = f"GW10_" + GW10_adata.obs.index

GW12_adata = adata[adata.obs['slice_code'] == 'B03607C4E6_WT2024071214941.h5ad'].copy()
GW12_adata.obs['stage'] = 'GW12'
GW12_adata.obs.index = f"GW12_" + GW12_adata.obs.index

GW13_adata = adata[adata.obs['slice_code'] == '43_A03590E1G4_WT202403310064.h5ad'].copy()
GW13_adata.obs['stage'] = 'GW13'
GW13_adata.obs.index = f"GW13_" + GW13_adata.obs.index

GW16_adata = adata[adata.obs['slice_code'] == 'B03618D3F6_WT202407152793.h5ad'].copy()
GW16_adata.obs['stage'] = 'GW16'
GW16_adata.obs.index = f"GW16_" + GW16_adata.obs.index

In [ ]:
combined_adata = ad.concat([GW10_adata, GW12_adata, GW13_adata, GW16_adata])
dyn.pp.select_genes_by_pearson_residuals(adata=combined_adata, n_top_genes=2000, inplace=True)
combined_hvg_adata = combined_adata[:, combined_adata.var["gene_highly_variable"]].copy()

GW10_adata = combined_hvg_adata[combined_hvg_adata.obs['stage'] == 'GW10']
GW12_adata = combined_hvg_adata[combined_hvg_adata.obs['stage'] == 'GW12']
GW13_adata = combined_hvg_adata[combined_hvg_adata.obs['stage'] == 'GW13']
GW16_adata = combined_hvg_adata[combined_hvg_adata.obs['stage'] == 'GW16']


|-----> gene selection on layer: X
|-----> extracting highly variable genes
|-----------> filtered out 4008 outlier genes


In [ ]:
mapped_adatas, mapping_pis = st.align.morpho_align(
    models=[GW10_adata.copy(), GW12_adata.copy(), GW13_adata.copy(), GW16_adata.copy()],
    spatial_key='align_spatial_2d',
    key_added='align_spatial_2d_morpho',
    device='0',
    verbose=True,

    # Nonrigid part
    beta=0.2,
    lambdaVF=1,
    K = 200,

    # input feature
    rep_layer=['X', 'dmt_leiden_annotation_0115'],
    rep_field=['layer', 'obs'],
    dissimilarity=['kl', 'label'],
    # label_transfer_dict=label_transfer_prior,
    probability_parameters=[0.2, None],

    # Acceleration and scalability
    sparse_calculation_mode=True,
    use_chunk=True,
    chunk_capacity=2,

    # Default setting for 3D spatiotemporal mapping
    partial_robust_level=1,
    return_mapping=True,
    nn_init=False,
    sigma2_init_scale=2,
    separate_scale=True,
)